# Diagnóstico de los modelos de recomendación

## Objetivo

Los experimentos realizados sobre el Modelo A permitieron mejorar el HR@10 desde 5,8 % hasta 7,2 %, quedando muy cerca del baseline de popularidad, que obtuvo 7,4 %.

Además, el Modelo A2 logró un NDCG@10 ligeramente superior al baseline.

Antes de desarrollar una nueva variante, se realiza un análisis de los resultados para comprender mejor el comportamiento del baseline y del modelo personalizado.

El objetivo de este diagnóstico es analizar:

- la concentración de las interacciones en los productos más populares;
- la diversidad de productos recomendados;
- las diferencias entre los aciertos del Modelo A2 y del baseline;
- si ambos enfoques cometen errores diferentes.

A partir de estos resultados se evaluará si existe una oportunidad concreta para desarrollar un modelo final que combine las fortalezas de ambos enfoques.

In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from IPython.display import display

PROJECT_ROOT = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "data" / "processed").is_dir()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró la carpeta data/processed"
    )

helper_dir = next(
    (
        p for p in [Path.cwd(), PROJECT_ROOT / "notebooks"]
        if (p / "sephora_common.py").exists()
    ),
    None
)

if helper_dir is None:
    raise FileNotFoundError(
        "No se encontró sephora_common.py"
    )

sys.path.insert(0, str(helper_dir))

import sephora_common as sc

N_VALIDACION = 500
N_TEST = 500
SEMILLA = 42

ctx = sc.load_context(
    N_VALIDACION,
    N_TEST,
    SEMILLA
)

print("Protocolo:", ctx.protocol_id[:12])
print("Usuarios entrenamiento:", ctx.train.author_id.nunique())
print("Productos del catálogo:", ctx.catalog.product_id.nunique())

Protocolo: 99fc64b0ea63
Usuarios entrenamiento: 502216
Productos del catálogo: 2420


In [3]:
# Nos quedamos únicamente con las interacciones positivas del entrenamiento
positivos_train = ctx.train[
    ctx.train["positive"] == True
].copy()

# Contamos cuántas interacciones positivas tiene cada producto
popularidad = (
    positivos_train
    .groupby("product_id")
    .size()
    .reset_index(name="interacciones_positivas")
    .sort_values(
        "interacciones_positivas",
        ascending=False
    )
    .reset_index(drop=True)
)

# Total de interacciones positivas
total_positivos = popularidad[
    "interacciones_positivas"
].sum()

# Calculamos cuánto concentran los productos más populares
resultados = []

for top_n in [10, 20, 50, 100]:
    
    positivos_top = popularidad.head(top_n)[
        "interacciones_positivas"
    ].sum()
    
    porcentaje = (
        positivos_top / total_positivos
    ) * 100
    
    resultados.append({
        "Top productos": f"Top {top_n}",
        "Interacciones positivas": positivos_top,
        "% del total": porcentaje
    })

concentracion = pd.DataFrame(resultados)

display(concentracion)

,Top productos,Interacciones positivas,% del total
0,Top 10,67111,7.318802
1,Top 20,110127,12.009920
2,Top 50,204029,22.250419
3,Top 100,310913,33.906673


In [4]:
# Productos más populares según el entrenamiento
ranking_popularidad = popularidad["product_id"].tolist()

# Productos objetivo de los 500 usuarios de test
targets_test = ctx.test["target"].astype(str)

resultados_test = []

for top_n in [10, 20, 50, 100]:
    
    productos_top = set(
        ranking_popularidad[:top_n]
    )
    
    cantidad_targets = targets_test.isin(
        productos_top
    ).sum()
    
    porcentaje = (
        cantidad_targets / len(targets_test)
    ) * 100
    
    resultados_test.append({
        "Grupo": f"Top {top_n}",
        "Targets dentro del grupo": cantidad_targets,
        "% de los 500 targets": porcentaje
    })

concentracion_test = pd.DataFrame(
    resultados_test
)

display(concentracion_test)

,Grupo,Targets dentro del grupo,% de los 500 targets
0,Top 10,37,7.4
1,Top 20,61,12.2
2,Top 50,122,24.4
3,Top 100,201,40.2


In [6]:
#Dónde se guardaron los resultados de los experimentos
print(ctx.output_dir)

c:\Users\Administrator\Documents\DATA SCIENCE\PF\sephora-recommender\artifacts\comparacion_4_modelos\99fc64b0ea63


In [7]:
# Ver los archivos guardados para este protocolo
archivos = sorted(ctx.output_dir.iterdir())

for archivo in archivos:
    print(archivo.name)

01_A2_perfil_popularidad.json
01_A2_perfil_popularidad_example_top10.csv
01_A2_perfil_popularidad_test_detail.csv
01_A3_perfil_popularidad_segmentada.json
01_A3_perfil_popularidad_segmentada_example_top10.csv
01_A3_perfil_popularidad_segmentada_test_detail.csv
01_A4_perfil_popularidad_contenido.json
01_A4_perfil_popularidad_contenido_example_top10.csv
01_A4_perfil_popularidad_contenido_test_detail.csv
01_vecinos_por_perfil.json
01_vecinos_por_perfil_example_top10.csv
01_vecinos_por_perfil_test_detail.csv
02_satisfaccion_por_segmento.json
02_satisfaccion_por_segmento_example_top10.csv
02_satisfaccion_por_segmento_test_detail.csv
03_similitud_ponderada.json
03_similitud_ponderada_example_top10.csv
03_similitud_ponderada_test_detail.csv
04_afinidad_supervisada.json
04_afinidad_supervisada_example_top10.csv
04_afinidad_supervisada_test_detail.csv
protocolo.json


In [8]:
# Cargar el detalle del test de A2
ruta_a2 = (
    ctx.output_dir
    / "01_A2_perfil_popularidad_test_detail.csv"
)

detalle_a2 = pd.read_csv(ruta_a2)

print("Filas:", len(detalle_a2))
print("Columnas:")
print(detalle_a2.columns.tolist())

display(detalle_a2.head())

Filas: 500
Columnas:
['author_id', 'target', 'hit', 'ndcg_at_10', 'n_recommendations', 'seconds', 'ids']


,author_id,target,hit,ndcg_at_10,n_recommendations,seconds,ids
0,11375150805,P472021,0,0.0,10,0.028065,"[""P420652"", ""P309308"", ""P423688"", ""P443352"", ""..."
1,5472040401,P472454,0,0.0,10,0.018143,"[""P420652"", ""P269122"", ""P126301"", ""P453816"", ""..."
2,22189028748,P415619,0,0.0,10,0.017966,"[""P420652"", ""P394639"", ""P248407"", ""P427421"", ""..."
3,6426650398,P392246,0,0.0,10,0.018443,"[""P420652"", ""P427417"", ""P394639"", ""P269122"", ""..."
4,21987520527,P444970,0,0.0,10,0.018701,"[""P420652"", ""P269122"", ""P126301"", ""P453816"", ""..."


In [9]:
# Top-10 del baseline de popularidad
top10_baseline = set(
    popularidad.head(10)["product_id"].astype(str)
)

# Copiamos el detalle de A2
comparacion = detalle_a2[
    ["author_id", "target", "hit"]
].copy()

comparacion["target"] = comparacion["target"].astype(str)

# Hit del baseline:
# es 1 cuando el target pertenece a los 10 productos más populares
comparacion["hit_baseline"] = (
    comparacion["target"]
    .isin(top10_baseline)
    .astype(int)
)

# Renombramos el hit de A2 para que quede claro
comparacion = comparacion.rename(
    columns={"hit": "hit_A2"}
)

# Clasificamos cada usuario según quién acertó
def clasificar_resultado(fila):
    
    if fila["hit_A2"] == 1 and fila["hit_baseline"] == 1:
        return "Ambos"
    
    elif fila["hit_A2"] == 1:
        return "Solo A2"
    
    elif fila["hit_baseline"] == 1:
        return "Solo baseline"
    
    else:
        return "Ninguno"


comparacion["resultado"] = comparacion.apply(
    clasificar_resultado,
    axis=1
)

resumen = (
    comparacion["resultado"]
    .value_counts()
    .reindex(
        ["Ambos", "Solo A2", "Solo baseline", "Ninguno"],
        fill_value=0
    )
    .reset_index()
)

resumen.columns = ["Resultado", "Usuarios"]

display(resumen)

print()
print("Hits A2:", comparacion["hit_A2"].sum())
print("Hits baseline:", comparacion["hit_baseline"].sum())

,Resultado,Usuarios
0,Ambos,26
1,Solo A2,10
2,Solo baseline,11
3,Ninguno,453



Hits A2: 36
Hits baseline: 37


In [10]:
# Casos donde A2 acierta y el baseline no
solo_a2 = comparacion[
    comparacion["resultado"] == "Solo A2"
].copy()

# Posición de cada producto en el ranking global de popularidad
ranking_posicion = {
    product_id: posicion
    for posicion, product_id in enumerate(
        ranking_popularidad,
        start=1
    )
}

# Agregamos la posición de popularidad del target
solo_a2["ranking_popularidad"] = (
    solo_a2["target"]
    .map(ranking_posicion)
)

# Agregamos información del producto para poder interpretarlo
columnas_producto = [
    columna
    for columna in [
        "product_id",
        "product_name",
        "brand_name",
        "secondary_category"
    ]
    if columna in ctx.catalog.columns
]

solo_a2 = solo_a2.merge(
    ctx.catalog[columnas_producto],
    left_on="target",
    right_on="product_id",
    how="left"
)

# Mostramos los 10 aciertos exclusivos de A2
columnas_mostrar = [
    columna
    for columna in [
        "target",
        "product_name",
        "brand_name",
        "secondary_category",
        "ranking_popularidad"
    ]
    if columna in solo_a2.columns
]

display(
    solo_a2[columnas_mostrar]
    .sort_values("ranking_popularidad")
)

,target,product_name,brand_name,secondary_category,ranking_popularidad
2,P411387,Superfood Antioxidant Cleanser,Youth To The People,Cleansers,11
7,P411387,Superfood Antioxidant Cleanser,Youth To The People,Cleansers,11
6,P441644,Mini Superfood Antioxidant Cleanser,Youth To The People,Mini Size,12
8,P441644,Mini Superfood Antioxidant Cleanser,Youth To The People,Mini Size,12
3,P411540,Cicapair Tiger Grass Color Correcting Treatmen...,Dr. Jart+,Moisturizers,18
4,P126301,Take The Day Off Cleansing Balm Makeup Remover,CLINIQUE,Cleansers,21
0,P480630,Mini Jet Lag Mask,Summer Fridays,Masks,29
1,P4016,Acne Control Clarifying Cleanser,Murad,Cleansers,77
5,P4016,Acne Control Clarifying Cleanser,Murad,Cleansers,77
9,P4016,Acne Control Clarifying Cleanser,Murad,Cleansers,77


In [12]:
# Casos exclusivos de A2 cuyo target ocupa
# la posición 77 en popularidad global
casos_77 = solo_a2[
    solo_a2["ranking_popularidad"] == 77
].copy()

# Igualamos el tipo de author_id para poder unir las tablas
casos_77["author_id"] = (
    casos_77["author_id"]
    .astype(str)
)

perfiles = ctx.profiles.reset_index().copy()

perfiles["author_id"] = (
    perfiles["author_id"]
    .astype(str)
)

# Unimos los casos con sus perfiles
perfiles_77 = casos_77[
    [
        "author_id",
        "target",
        "product_name",
        "ranking_popularidad"
    ]
].merge(
    perfiles,
    on="author_id",
    how="left"
)

# Mostramos las características utilizadas por A2
columnas_mostrar = [
    "author_id",
    "target",
    "product_name",
    "skin_type",
    "skin_tone",
    "eye_color",
    "hair_color",
    "ranking_popularidad"
]

display(
    perfiles_77[columnas_mostrar]
)

,author_id,target,product_name,skin_type,skin_tone,eye_color,hair_color,ranking_popularidad
0,2352148499,P4016,Acne Control Clarifying Cleanser,combination,fair,Unknown,Unknown,77
1,1635532180,P4016,Acne Control Clarifying Cleanser,oily,mediumTan,Unknown,Unknown,77
2,1325461765,P4016,Acne Control Clarifying Cleanser,combination,fair,Unknown,Unknown,77


In [13]:
# Producto que A2 acertó 3 veces
producto_77 = casos_77["target"].iloc[0]

# Perfiles que aparecieron en esos tres casos
perfiles_analizar = [
    ("combination", "fair"),
    ("oily", "mediumTan")
]

resultados_segmentos = []

for skin_type, skin_tone in perfiles_analizar:

    # Usuarios de entrenamiento con ese perfil
    usuarios_segmento = ctx.profiles[
        (ctx.profiles["skin_type"] == skin_type) &
        (ctx.profiles["skin_tone"] == skin_tone)
    ].index

    # Interacciones positivas de esos usuarios
    positivos_segmento = ctx.train[
        (ctx.train["author_id"].isin(usuarios_segmento)) &
        (ctx.train["positive"] == True)
    ]

    # Ranking de productos dentro del segmento
    ranking_segmento = (
        positivos_segmento
        .groupby("product_id")
        .size()
        .sort_values(ascending=False)
        .reset_index(name="interacciones_positivas")
    )

    ranking_segmento["posicion"] = (
        range(1, len(ranking_segmento) + 1)
    )

    # Buscamos nuestro producto #77
    producto_en_segmento = ranking_segmento[
        ranking_segmento["product_id"] == producto_77
    ]

    if not producto_en_segmento.empty:

        resultados_segmentos.append({
            "skin_type": skin_type,
            "skin_tone": skin_tone,
            "usuarios_segmento": len(usuarios_segmento),
            "ranking_segmento": int(
                producto_en_segmento["posicion"].iloc[0]
            ),
            "positivos_producto": int(
                producto_en_segmento[
                    "interacciones_positivas"
                ].iloc[0]
            )
        })

    else:

        resultados_segmentos.append({
            "skin_type": skin_type,
            "skin_tone": skin_tone,
            "usuarios_segmento": len(usuarios_segmento),
            "ranking_segmento": None,
            "positivos_producto": 0
        })

resultado_producto_77 = pd.DataFrame(
    resultados_segmentos
)

print("Ranking global del producto:", 77)

display(resultado_producto_77)

Ranking global del producto: 77


,skin_type,skin_tone,usuarios_segmento,ranking_segmento,positivos_producto
0,combination,fair,51818,118,186
1,oily,mediumTan,4622,11,44


In [14]:
# Copiamos los 10 casos exclusivos de A2
analisis_a2 = solo_a2.copy()

# Igualamos el tipo de author_id
analisis_a2["author_id"] = (
    analisis_a2["author_id"]
    .astype(str)
)

perfiles = ctx.profiles.reset_index().copy()

perfiles["author_id"] = (
    perfiles["author_id"]
    .astype(str)
)

# Agregamos skin_type y skin_tone de cada usuario
analisis_a2 = analisis_a2.merge(
    perfiles[
        ["author_id", "skin_type", "skin_tone"]
    ],
    on="author_id",
    how="left"
)

resultados_perfil = []

for _, fila in analisis_a2.iterrows():

    skin_type = fila["skin_type"]
    skin_tone = fila["skin_tone"]
    producto = fila["target"]

    # Usuarios de entrenamiento con el mismo perfil
    usuarios_segmento = ctx.profiles[
        (ctx.profiles["skin_type"] == skin_type) &
        (ctx.profiles["skin_tone"] == skin_tone)
    ].index

    # Interacciones positivas de ese segmento
    positivos_segmento = ctx.train[
        (ctx.train["author_id"].isin(usuarios_segmento)) &
        (ctx.train["positive"] == True)
    ]

    # Ranking de productos dentro del segmento
    ranking_segmento = (
        positivos_segmento
        .groupby("product_id")
        .size()
        .sort_values(ascending=False)
        .reset_index(name="positivos_segmento")
    )

    ranking_segmento["ranking_segmento"] = (
        range(1, len(ranking_segmento) + 1)
    )

    producto_segmento = ranking_segmento[
        ranking_segmento["product_id"] == producto
    ]

    if not producto_segmento.empty:
        posicion_segmento = int(
            producto_segmento["ranking_segmento"].iloc[0]
        )
    else:
        posicion_segmento = None

    resultados_perfil.append({
        "target": producto,
        "skin_type": skin_type,
        "skin_tone": skin_tone,
        "ranking_global": fila["ranking_popularidad"],
        "ranking_segmento": posicion_segmento
    })


comparacion_ranking = pd.DataFrame(
    resultados_perfil
)

# Diferencia positiva = el producto sube dentro del perfil
comparacion_ranking["mejora_ranking"] = (
    comparacion_ranking["ranking_global"]
    - comparacion_ranking["ranking_segmento"]
)

display(
    comparacion_ranking.sort_values(
        "mejora_ranking",
        ascending=False
    )
)

,target,skin_type,skin_tone,ranking_global,ranking_segmento,mejora_ranking
5,P4016,oily,mediumTan,77,11,66
0,P480630,dry,mediumTan,29,5,24
3,P411540,oily,fair,18,11,7
2,P411387,normal,fair,11,6,5
8,P441644,normal,light,12,7,5
6,P441644,combination,light,12,11,1
7,P411387,combination,mediumTan,11,13,-2
4,P126301,combination,fair,21,35,-14
1,P4016,combination,fair,77,118,-41
9,P4016,combination,fair,77,118,-41


In [15]:
print(detalle_a2.columns.tolist())

['author_id', 'target', 'hit', 'ndcg_at_10', 'n_recommendations', 'seconds', 'ids']


In [16]:
import ast

# Convertimos la columna ids de texto a lista
listas_recomendaciones = detalle_a2["ids"].apply(
    ast.literal_eval
)

# Cantidad de usuarios analizados
n_usuarios = len(listas_recomendaciones)

# Productos distintos recomendados entre todos los usuarios
productos_recomendados = set()

for lista in listas_recomendaciones:
    productos_recomendados.update(lista)

print("Usuarios analizados:", n_usuarios)
print(
    "Productos distintos recomendados por A2:",
    len(productos_recomendados)
)

print(
    "Productos disponibles en el catálogo:",
    ctx.catalog["product_id"].nunique()
)

cobertura_catalogo = (
    len(productos_recomendados)
    / ctx.catalog["product_id"].nunique()
) * 100

print(
    "Cobertura del catálogo:",
    round(cobertura_catalogo, 2),
    "%"
)

Usuarios analizados: 500
Productos distintos recomendados por A2: 81
Productos disponibles en el catálogo: 2420
Cobertura del catálogo: 3.35 %


In [17]:
# Top-10 global utilizado por el baseline
top10_global = set(
    popularidad.head(10)["product_id"].astype(str)
)

coincidencias = []

for lista in listas_recomendaciones:
    
    lista = [str(producto) for producto in lista]
    
    cantidad_top10 = sum(
        producto in top10_global
        for producto in lista
    )
    
    coincidencias.append(cantidad_top10)

coincidencias = pd.Series(coincidencias)

print(
    "Promedio de productos del baseline dentro del Top-10 de A2:",
    round(coincidencias.mean(), 2),
    "de 10"
)

print(
    "Mínimo:",
    coincidencias.min()
)

print(
    "Máximo:",
    coincidencias.max()
)

print()
print("Distribución:")
display(
    coincidencias
    .value_counts()
    .sort_index()
    .rename_axis("Productos del Top-10 global")
    .reset_index(name="Usuarios")
)

Promedio de productos del baseline dentro del Top-10 de A2: 6.37 de 10
Mínimo: 3
Máximo: 9

Distribución:


,Productos del Top-10 global,Usuarios
0,3,5
1,4,21
2,5,68
3,6,188
4,7,126
5,8,91
6,9,1
